## Tools Used
- Python (Pandas, NumPy)
- Matplotlib & Seaborn
- Folium (for maps)

# King County Housing Analysis

## Objective
The purpose of this analysis is to explore the King County housing dataset, identify the main factors affecting house prices, and provide practical insights for buyers, sellers, and investors.

In [ ]:
# Imports & Setup
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter

import folium
from folium.plugins import MarkerCluster

from geopy.distance import geodesic

# Plot settings
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.facecolor": "white",
    "axes.edgecolor": "black"
})

plt.rcParams["figure.facecolor"] = "w"

pd.plotting.register_matplotlib_converters()
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# Load dataset
df = pd.read_csv('data/merged_dataset.csv')

: 

## Dataset Overview
This dataset contains housing sales data, including price, property characteristics, condition, and location-related features. The analysis begins with a general inspection of the data structure and quality.

In [ ]:
df.head()
df.shape
df.info()
df.describe()

## Defining Center vs Outskirts

To analyze how location affects housing prices, I divided properties into two groups: "center" and "outskirts". 

This classification is based on a latitude threshold, which acts as a simple way to separate more central areas from more peripheral ones.

In [ ]:
# Define latitude threshold
center_latitude_threshold = 47.6

# Create location categories
df['location'] = np.where(df['lat'] >= center_latitude_threshold, 'center', 'outskirts')

# Count properties
location_counts = df['location'].value_counts()

location_counts

**Insight:** The dataset is divided into central and outer areas, allowing for a clearer comparison of housing patterns. This grouping enables further analysis of whether properties in central locations differ in price compared to those in the outskirts.

In [ ]:
df.groupby('location')['price'].mean()

**Insight:** Properties classified as "center" tend to have higher average prices compared to those in the outskirts, suggesting that location plays a significant role in determining property value.

## Relationship Between Living Space and Price

To understand how property size influences price, I analyzed the relationship between the average living area of nearby properties (`sqft_living15`) and house prices.

This helps identify whether larger homes are generally associated with higher prices.

In [ ]:
# Correlation analysis
correlation = df[['sqft_living15', 'price']].corr()
correlation

**Insight:** The correlation coefficient indicates the strength and direction of the relationship between living space and price. A positive value suggests that larger properties tend to have higher prices.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='sqft_living15', y='price', data=df)
plt.title('sqft_living15 vs Price')
plt.xlabel('sqft_living15')
plt.ylabel('Price')
plt.show()

**Insight:** The scatter plot shows a clear upward trend, indicating a positive relationship between living space and price. However, there is noticeable variability, suggesting that other factors (such as location or amenities) also influence pricing.

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(x='sqft_living15', y='price', data=df)
plt.title('Regression: sqft_living15 vs Price')
plt.xlabel('sqft_living15')
plt.ylabel('Price')
plt.show()

**Key Insight:** There is a strong positive relationship between living space and house price. As the size of a property increases, its price generally rises. 

However, the spread of points suggests that size alone does not determine value—location and other features likely play an important role.

In [ ]:
correlation_value = correlation.loc['sqft_living15', 'price']
correlation_value

The correlation coefficient of approximately X indicates a moderate/strong positive relationship between living space and price.

## Waterfront Properties Analysis

Waterfront properties are typically considered premium real estate. To better understand their impact on pricing, I filtered properties with waterfront access and analyzed the top 10% most expensive among them.

This helps highlight the characteristics of high-end waterfront homes.

In [ ]:
# Filter waterfront properties
waterfront_houses = df[df['waterfront'] == 1]

# Select top 10% most expensive waterfront properties
top_10_percent_waterfront = waterfront_houses.nlargest(
    int(0.10 * len(waterfront_houses)), 'price'
)

top_10_percent_waterfront

**Insight:** Waterfront properties represent a premium segment of the housing market. By focusing on the top 10%, we can observe the highest-value properties, which are likely influenced by both location and luxury features.

These properties significantly exceed average prices, reinforcing the strong impact of waterfront access on property value.

In [ ]:
# Compare average prices
waterfront_avg = waterfront_houses['price'].mean()
overall_avg = df['price'].mean()

waterfront_avg, overall_avg

**Key Insight:** The average price of waterfront properties is significantly higher than the overall average, confirming that waterfront access is a major driver of property value.

## Focusing on the Core Housing Market

To better understand typical housing patterns, I excluded the top and bottom 20% of property prices. 

This allows the analysis to focus on the central portion of the market, reducing the influence of extreme values and providing a clearer view of general trends.

In [ ]:
# Define thresholds
lower_threshold = df['price'].quantile(0.2)
upper_threshold = df['price'].quantile(0.8)

# Filter middle 60%
middle_market = df[
    (df['price'] >= lower_threshold) &
    (df['price'] <= upper_threshold)
]

middle_market.shape

**Insight:** By focusing on the middle 60% of properties, we reduce the impact of extreme high-end and low-end values. This provides a more representative view of the general housing market.

In [ ]:
df['price'].mean(), middle_market['price'].mean()

**Key Insight:** The average price in the middle market differs from the overall average, showing how extreme values (especially luxury properties) can skew general statistics.

This filtered dataset allows for a clearer comparison of factors such as location, living space, and property features without the distortion caused by extreme price values.

## Housing Market Activity Over Time

To understand how housing market activity changes over time, I grouped transactions by month and counted the number of entries.

This helps identify periods of higher or lower market activity and potential seasonal patterns.

In [ ]:
# Convert 'date' to datetime
df['date'] = pd.to_datetime(df['date'])

# Extract year-month
df['year_month'] = df['date'].dt.to_period('M')

# Count transactions per month
monthly_counts = df.groupby('year_month').size().reset_index(name='count')

# Plot
plt.figure(figsize=(15, 6))
plt.bar(monthly_counts['year_month'].astype(str), monthly_counts['count'])

plt.xlabel('Year-Month')
plt.ylabel('Number of Sales')
plt.title('Housing Market Activity Over Time')

plt.xticks(rotation=45, ha='right')
plt.show()

**Insight:** The distribution of transactions over time shows variations in market activity. Certain months have higher numbers of property sales, indicating periods of increased demand or market activity.

## Geographic Distribution of Prices

To better understand how location influences housing prices, I created a scatter plot of property locations, with each point color-coded based on price.

This allows for a visual comparison of price variations across different geographic areas.

In [ ]:
plt.figure(figsize=(10, 8))

plt.scatter(
    df['long'], 
    df['lat'], 
    c=df['price'], 
    cmap='viridis', 
    s=10, 
    alpha=0.5
)

plt.colorbar(label='Price')

plt.title('Geographic Distribution of House Prices')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

plt.show()

**Key Insight:** Clusters of high-value properties likely correspond to desirable neighborhoods or waterfront areas, reinforcing earlier findings about the impact of location and amenities on property value.

## Monthly Average Sale Prices

To analyze overall price trends, I calculated the average house price for each month.

This provides a clearer view of general market behavior by reducing the impact of extreme high-value or low-value transactions.

In [ ]:
# Ensure datetime
df['date'] = pd.to_datetime(df['date'])

# Create year-month
df['year_month'] = df['date'].dt.to_period('M')

# Calculate monthly average prices
monthly_mean_prices = (
    df.groupby('year_month')['price']
    .mean()
    .reset_index()
)

# Plot
plt.figure(figsize=(15, 6))

plt.plot(
    monthly_mean_prices['year_month'].astype(str),
    monthly_mean_prices['price'],
    color='green',
    linewidth=2,
    label='Average Prices'
)

plt.xlabel('Year-Month')
plt.ylabel('Average Price')
plt.title('Average Sales Prices Over the Months')
plt.xticks(rotation=45, ha='right')
plt.legend()

plt.tight_layout()
plt.show()

**Insight:** The average price over time shows moderate fluctuations, suggesting that while individual transactions vary significantly, the overall market remains relatively stable.

This reinforces the importance of considering aggregated trends rather than individual sales.

## Waterfront Properties by Condition

To further explore premium properties, I analyzed how property condition affects prices among waterfront homes.

This helps identify whether better-maintained properties command higher prices within this high-value segment.

In [ ]:
waterfront_df = df[df['waterfront'] == 1]

waterfront_df = waterfront_df.dropna(subset=['price', 'condition'])

In [ ]:
waterfront_df.groupby('condition')['price'].mean()

In [ ]:
plt.figure(figsize=(8, 5))

waterfront_df.groupby('condition')['price'].mean().plot(kind='bar')

plt.title('Average Price by Condition (Waterfront Properties)')
plt.xlabel('Condition')
plt.ylabel('Average Price')

plt.show()

**Insight:** Waterfront properties with higher condition ratings tend to have higher average prices, suggesting that both location and property quality contribute to overall value.